# Reproducing Isolation Forest

This notebook reproduces the main experiment of the paper: the AUC and processing time of iForest on the benchmark datasets.

The datasets must be in a the same folder as the notebook with the name DATA

In [1]:
import time
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.ensemble import IsolationForest
from sklearn.metrics import roc_auc_score

## Parameters

Same settings as the paper. The height limit is set automatically by scikit-learn.

In [ ]:
DATA = Path("data")   # folder with the .npz files

n_trees = 100   #number of trees
psi = 256       #subsample size per tree, drawn without replacement
runs = 10       #one random seed per run (0 to 9)

## Datasets and results reported in the paper (Table 3)

In [3]:
datasets = {
    "16_http":        ("Http",        1.00, 15.58),
    "10_cover":       ("ForestCover", 0.88, 16.33),
    "34_smtp":        ("Smtp",        0.88, 2.72),
    "32_shuttle":     ("Shuttle",     1.00, 3.13),
    "23_mammography": ("Mammography", 0.86, 0.66),
    "2_annthyroid":   ("Annthyroid",  0.82, 0.51),
    "30_satellite":   ("Satellite",   0.71, 1.63),
    "29_Pima":        ("Pima",        0.67, 0.28),
    "4_breastw":      ("Breastw",     0.99, 0.28),
    "18_Ionosphere":  ("Ionosphere",  0.85, 0.48),
}

## One run of iForest

The model is trained without labels. The labels are only used to compute the AUC.

In [4]:
def run_once(X, y, seed):
    start = time.perf_counter()
    model = IsolationForest(n_estimators=n_trees, max_samples=psi, random_state=seed)
    model.fit(X)                       
    scores = -model.score_samples(X)  
    elapsed = time.perf_counter() - start
    return roc_auc_score(y, scores), elapsed

## Run the experiment

In [9]:
rows = []
for file, (name, paper_auc, paper_time) in datasets.items():
    data = np.load(DATA / f"{file}.npz")
    X, y = data["X"], data["y"]
    aucs, times = zip(*(run_once(X, y, seed) for seed in range(runs)))
    rows.append({
        "dataset": name,
        "n": len(X),
        "anomalies_%": round(100 * y.mean(), 2),
        "paper_AUC": paper_auc,
        "our_AUC": round(np.mean(aucs), 2),
        "paper_time_s": paper_time,
        "our_time_s": round(np.mean(times), 2),
    })

results = pd.DataFrame(rows)
results.to_csv("results_table3.csv", index=False)
results

,dataset,n,anomalies_%,paper_AUC,our_AUC,paper_time_s,our_time_s
0,Http,567498,0.39,1.00,1.00,15.58,1.96
1,ForestCover,286048,0.96,0.88,0.88,16.33,0.89
2,Smtp,95156,0.03,0.88,0.90,2.72,0.39
3,Shuttle,49097,7.15,1.00,1.00,3.13,0.25
4,Mammography,11183,2.32,0.86,0.86,0.66,0.12
5,Annthyroid,7200,7.42,0.82,0.82,0.51,0.10
6,Satellite,6435,31.64,0.71,0.70,1.63,0.10
7,Pima,768,34.90,0.67,0.67,0.28,0.07
8,Breastw,683,34.99,0.99,0.99,0.28,0.07
9,Ionosphere,351,35.90,0.85,0.85,0.48,0.07
